# MeteoHub Adriatic — Aggregazione per Stazione (ultima lettura)

Trasforma `adriatic_coast_stations.csv` da formato **long** (una riga per lettura: stazione × variabile × timestamp) a formato **wide** (una riga per stazione, con l'ultimo valore disponibile per ciascuna variabile).

**Input:** `adriatic_coast_stations.csv` (stessa cartella del notebook)
**Output:** `adriatic_coast_stations_aggregated.csv` — atteso 1 riga per stazione (~189)

## Logica di aggregazione
Per ogni stazione e ogni variabile (`TARIA2M`, `PRESS`, `VV`, `PREC`), si prende il valore con il timestamp `dt` più recente — cioè l'ultima lettura disponibile nella finestra di 3 giorni scaricata.

## 0. Imports and Configuration

In [1]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

INPUT_CSV  = 'adriatic_coast_stations.csv'
OUTPUT_CSV = 'adriatic_coast_stations_aggregated.csv'

# Colonne che identificano univocamente la stazione (restano fisse per ogni gruppo)
STATION_COLS = ['station_id', 'sensor_name', 'lat', 'lon',
                'codseqst', 'point', 'quota', 'gestore', 'provincia']

# Variabili da pivotare in colonne separate
SENSOR_CODES = ['TARIA2M', 'PRESS', 'VV', 'PREC']

print(f'Input : {INPUT_CSV}')
print(f'Output: {OUTPUT_CSV}')

Input : adriatic_coast_stations.csv
Output: adriatic_coast_stations_aggregated.csv


## 1. Load Data

In [2]:
df = pd.read_csv(INPUT_CSV)
df['dt'] = pd.to_datetime(df['dt'], utc=True)

print(f'Total readings (long format) : {len(df)}')
print(f'Unique stations               : {df["station_id"].nunique()}')
print(f'Sensor codes present          : {sorted(df["sensor_code"].unique())}')
print(f'Time range                    : {df["dt"].min()} → {df["dt"].max()}')
print()
df.head()

Total readings (long format) : 33347
Unique stations               : 122
Sensor codes present          : ['PREC', 'PRESS', 'TARIA2M', 'VV']
Time range                    : 2026-06-18 16:46:00+00:00 → 2026-06-21 16:10:00+00:00



,station_id,sensor_name,sensor_code,quantity,unit,dt,value,lat,lon,codseqst,point,quota,aggiornamento,gestore,provincia
0,MH_dpcn-marche_Ancona_Regione,Ancona Regione,PREC,Precipitazione,mm,2026-06-18 17:00:00+00:00,0.0,43.6103,13.5083,Ancona Regione,"{""type"": ""Point"", ""coordinates"": [13.5083, 43....",47.0,2026-06-21T16:46:01.341950+00:00,MeteoHub_dpcn-marche,NaN
1,MH_dpcn-marche_Ancona_Regione,Ancona Regione,PRESS,Pressione atmosferica,hPa,2026-06-18 17:00:00+00:00,1008.0,43.6103,13.5083,Ancona Regione,"{""type"": ""Point"", ""coordinates"": [13.5083, 43....",47.0,2026-06-21T16:46:01.341950+00:00,MeteoHub_dpcn-marche,NaN
2,MH_dpcn-marche_Ancona_Regione,Ancona Regione,TARIA2M,Temperatura aria a 2m,°C,2026-06-18 17:00:00+00:00,29.9,43.6103,13.5083,Ancona Regione,"{""type"": ""Point"", ""coordinates"": [13.5083, 43....",47.0,2026-06-21T16:46:01.341950+00:00,MeteoHub_dpcn-marche,NaN
3,MH_dpcn-marche_Ancona_Regione,Ancona Regione,VV,Velocità vento,m/s,2026-06-18 17:00:00+00:00,2.8,43.6103,13.5083,Ancona Regione,"{""type"": ""Point"", ""coordinates"": [13.5083, 43....",47.0,2026-06-21T16:46:01.341950+00:00,MeteoHub_dpcn-marche,NaN
4,MH_dpcn-marche_Ancona_Regione,Ancona Regione,PREC,Precipitazione,mm,2026-06-18 17:15:00+00:00,0.0,43.6103,13.5083,Ancona Regione,"{""type"": ""Point"", ""coordinates"": [13.5083, 43....",47.0,2026-06-21T16:46:01.341950+00:00,MeteoHub_dpcn-marche,NaN


## 2. Select Last Reading per Station × Sensor

Ordina per timestamp e tiene solo l'ultima riga per ogni combinazione `(station_id, sensor_code)`.

In [3]:
df_sorted = df.sort_values('dt')

last_readings = (
    df_sorted
    .groupby(['station_id', 'sensor_code'], as_index=False)
    .last()
)

print(f'Rows after taking last reading per (station, sensor): {len(last_readings)}')
print(f'Unique stations: {last_readings["station_id"].nunique()}')
print(f'Unique sensors per station — expected up to {len(SENSOR_CODES)}:')
print(last_readings.groupby('station_id')['sensor_code'].nunique().value_counts()
      .rename('n_stations').to_string())
last_readings.head(8)

Rows after taking last reading per (station, sensor): 248
Unique stations: 122
Unique sensors per station — expected up to 4:
sensor_code
2    83
1    23
4    11
3     5


,station_id,sensor_code,sensor_name,quantity,unit,dt,value,lat,lon,codseqst,point,quota,aggiornamento,gestore,provincia
0,MH_dpcn-marche_Acqualagna,PREC,Acqualagna,Precipitazione,mm,2026-06-21 16:00:00+00:00,0.0,43.62819,12.68448,Acqualagna,"{""type"": ""Point"", ""coordinates"": [12.68448, 43...",196.0,2026-06-21T16:46:01.411638+00:00,MeteoHub_dpcn-marche,NaN
1,MH_dpcn-marche_Acqualagna,TARIA2M,Acqualagna,Temperatura aria a 2m,°C,2026-06-21 16:00:00+00:00,30.8,43.62819,12.68448,Acqualagna,"{""type"": ""Point"", ""coordinates"": [12.68448, 43...",196.0,2026-06-21T16:46:01.411638+00:00,MeteoHub_dpcn-marche,NaN
2,MH_dpcn-marche_Agugliano,PREC,Agugliano,Precipitazione,mm,2026-06-21 16:00:00+00:00,0.0,43.54330,13.38060,Agugliano,"{""type"": ""Point"", ""coordinates"": [13.3806, 43....",141.0,2026-06-21T16:46:01.625396+00:00,MeteoHub_dpcn-marche,NaN
3,MH_dpcn-marche_Amandola,PREC,Amandola,Precipitazione,mm,2026-06-21 16:00:00+00:00,0.0,42.97638,13.35133,Amandola,"{""type"": ""Point"", ""coordinates"": [13.35133, 42...",0.0,2026-06-21T16:46:01.524236+00:00,MeteoHub_dpcn-marche,NaN
4,MH_dpcn-marche_Amandola,TARIA2M,Amandola,Temperatura aria a 2m,°C,2026-06-21 16:00:00+00:00,28.3,42.97638,13.35133,Amandola,"{""type"": ""Point"", ""coordinates"": [13.35133, 42...",0.0,2026-06-21T16:46:01.524236+00:00,MeteoHub_dpcn-marche,NaN
5,MH_dpcn-marche_Ancona_Regione,PREC,Ancona Regione,Precipitazione,mm,2026-06-21 16:00:00+00:00,0.0,43.61030,13.50830,Ancona Regione,"{""type"": ""Point"", ""coordinates"": [13.5083, 43....",47.0,2026-06-21T16:46:01.341950+00:00,MeteoHub_dpcn-marche,NaN
6,MH_dpcn-marche_Ancona_Regione,PRESS,Ancona Regione,Pressione atmosferica,hPa,2026-06-21 16:00:00+00:00,1011.0,43.61030,13.50830,Ancona Regione,"{""type"": ""Point"", ""coordinates"": [13.5083, 43....",47.0,2026-06-21T16:46:01.341950+00:00,MeteoHub_dpcn-marche,NaN
7,MH_dpcn-marche_Ancona_Regione,TARIA2M,Ancona Regione,Temperatura aria a 2m,°C,2026-06-21 16:00:00+00:00,28.5,43.61030,13.50830,Ancona Regione,"{""type"": ""Point"", ""coordinates"": [13.5083, 43....",47.0,2026-06-21T16:46:01.341950+00:00,MeteoHub_dpcn-marche,NaN


## 3. Pivot to Wide Format — One Row per Station

Ogni `sensor_code` diventa due colonne: `{CODE}_value` (l'ultimo valore) e `{CODE}_dt` (il timestamp di quella lettura, utile perché variabili diverse possono avere l'ultima lettura in momenti diversi).

In [4]:
# Pivot dei valori
pivot_values = last_readings.pivot(index='station_id', columns='sensor_code', values='value')
pivot_values.columns = [f'{c}_value' for c in pivot_values.columns]

# Pivot dei timestamp (per sapere quando è stata presa ciascuna lettura)
pivot_times = last_readings.pivot(index='station_id', columns='sensor_code', values='dt')
pivot_times.columns = [f'{c}_dt' for c in pivot_times.columns]

# Metadati stazione — invarianti, basta il primo valore per stazione
station_meta = (
    df.drop_duplicates('station_id')
    .set_index('station_id')[STATION_COLS[1:]]   # escludiamo station_id, già indice
)

# Unione finale
wide = station_meta.join(pivot_values).join(pivot_times).reset_index()

# Riordina colonne: metadati, poi value/dt alternati per sensore
ordered_cols = ['station_id'] + STATION_COLS[1:]
for code in SENSOR_CODES:
    if f'{code}_value' in wide.columns:
        ordered_cols += [f'{code}_value', f'{code}_dt']
wide = wide[[c for c in ordered_cols if c in wide.columns]]

print(f'Final shape: {wide.shape}  (expected ~189 rows, one per station)')
wide.head(10)

Final shape: (122, 17)  (expected ~189 rows, one per station)


,station_id,sensor_name,lat,lon,codseqst,point,quota,gestore,provincia,TARIA2M_value,TARIA2M_dt,PRESS_value,PRESS_dt,VV_value,VV_dt,PREC_value,PREC_dt
0,MH_dpcn-marche_Ancona_Regione,Ancona Regione,43.61030,13.50830,Ancona Regione,"{""type"": ""Point"", ""coordinates"": [13.5083, 43....",47.0,MeteoHub_dpcn-marche,NaN,28.5,2026-06-21 16:00:00+00:00,1011.0,2026-06-21 16:00:00+00:00,1.6,2026-06-21 16:00:00+00:00,0.0,2026-06-21 16:00:00+00:00
1,MH_dpcn-marche_Sassotetto,Sassotetto,43.00670,13.24170,Sassotetto,"{""type"": ""Point"", ""coordinates"": [13.2417, 43....",1344.0,MeteoHub_dpcn-marche,NaN,20.6,2026-06-21 16:00:00+00:00,876.0,2026-06-21 16:00:00+00:00,0.7,2026-06-21 16:00:00+00:00,0.0,2026-06-21 16:00:00+00:00
2,MH_dpcn-marche_Mozzano,Mozzano,42.84965,13.53753,Mozzano,"{""type"": ""Point"", ""coordinates"": [13.53753, 42...",-1.0,MeteoHub_dpcn-marche,NaN,31.1,2026-06-21 16:00:00+00:00,NaN,NaT,NaN,NaT,0.0,2026-06-21 16:00:00+00:00
3,MH_dpcn-marche_Sibilla,Sibilla,42.89500,13.27000,Sibilla,"{""type"": ""Point"", ""coordinates"": [13.27, 42.895]}",1676.0,MeteoHub_dpcn-marche,NaN,18.5,2026-06-21 16:00:00+00:00,838.0,2026-06-21 16:00:00+00:00,1.9,2026-06-21 16:00:00+00:00,NaN,NaT
4,MH_dpcn-marche_Monte_Nerone,Monte Nerone,43.55328,12.52087,Monte Nerone,"{""type"": ""Point"", ""coordinates"": [12.52087, 43...",0.0,MeteoHub_dpcn-marche,NaN,20.8,2026-06-21 16:00:00+00:00,NaN,NaT,3.0,2026-06-21 16:00:00+00:00,0.0,2026-06-21 16:00:00+00:00
5,MH_dpcn-marche_Pennabilli,Pennabilli,43.81972,12.27417,Pennabilli,"{""type"": ""Point"", ""coordinates"": [12.27417, 43...",589.0,MeteoHub_dpcn-marche,NaN,26.7,2026-06-21 16:00:00+00:00,950.5,2026-06-21 16:00:00+00:00,1.6,2026-06-21 16:00:00+00:00,NaN,NaT
6,MH_dpcn-marche_Urbino,Urbino,43.72330,12.63690,Urbino,"{""type"": ""Point"", ""coordinates"": [12.6369, 43....",384.0,MeteoHub_dpcn-marche,NaN,28.7,2026-06-21 16:00:00+00:00,966.0,2026-06-21 16:00:00+00:00,0.7,2026-06-21 16:00:00+00:00,0.0,2026-06-21 16:00:00+00:00
7,MH_dpcn-marche_Camerino,Camerino,43.14610,13.06610,Camerino,"{""type"": ""Point"", ""coordinates"": [13.0661, 43....",510.0,MeteoHub_dpcn-marche,NaN,28.6,2026-06-21 16:00:00+00:00,959.0,2026-06-21 16:00:00+00:00,1.8,2026-06-21 16:00:00+00:00,0.0,2026-06-21 16:00:00+00:00
8,MH_dpcn-marche_San_Benedetto,San Benedetto,42.93330,13.88920,San Benedetto,"{""type"": ""Point"", ""coordinates"": [13.8892, 42....",1.0,MeteoHub_dpcn-marche,NaN,28.2,2026-06-21 16:00:00+00:00,1021.0,2026-06-21 16:00:00+00:00,1.3,2026-06-21 16:00:00+00:00,0.0,2026-06-21 16:00:00+00:00
9,MH_dpcn-marche_Grottazzolina,Grottazzolina,43.10610,13.59860,Grottazzolina,"{""type"": ""Point"", ""coordinates"": [13.5986, 43....",195.0,MeteoHub_dpcn-marche,NaN,29.6,2026-06-21 16:00:00+00:00,NaN,NaT,NaN,NaT,0.0,2026-06-21 16:00:00+00:00


## 4. Coverage Check — Missing Sensors per Station

In [5]:
value_cols = [f'{c}_value' for c in SENSOR_CODES if f'{c}_value' in wide.columns]

print('Missing values per sensor (stations without that sensor):')
for col in value_cols:
    n_missing = wide[col].isna().sum()
    pct = 100 * n_missing / len(wide)
    print(f'  {col:14s}: {n_missing:3d} / {len(wide)} missing  ({pct:.1f}%)')

print(f'\nStations with ALL {len(value_cols)} sensors  : {wide[value_cols].notna().all(axis=1).sum()}')
print(f'Stations with at least 1 sensor     : {wide[value_cols].notna().any(axis=1).sum()}')
print(f'Stations with 0 sensors (empty rows): {wide[value_cols].isna().all(axis=1).sum()}')

Missing values per sensor (stations without that sensor):
  TARIA2M_value :  21 / 122 missing  (17.2%)
  PRESS_value   : 107 / 122 missing  (87.7%)
  VV_value      : 106 / 122 missing  (86.9%)
  PREC_value    :   6 / 122 missing  (4.9%)

Stations with ALL 4 sensors  : 11
Stations with at least 1 sensor     : 122
Stations with 0 sensors (empty rows): 0


## 5. Quick Sanity Check — Value Ranges

In [6]:
print('Value ranges per variable (physical plausibility check):')
for col in value_cols:
    s = wide[col].dropna()
    if len(s) > 0:
        print(f'  {col:14s}: min={s.min():.2f}  max={s.max():.2f}  '
              f'mean={s.mean():.2f}  n={len(s)}')

Value ranges per variable (physical plausibility check):
  TARIA2M_value : min=17.70  max=34.70  mean=28.52  n=101
  PRESS_value   : min=821.00  max=1022.00  mean=926.83  n=15
  VV_value      : min=0.70  max=5.50  mean=1.79  n=16
  PREC_value    : min=0.00  max=0.00  mean=0.00  n=116


## 6. Export

In [7]:
wide.to_csv(OUTPUT_CSV, index=False)

print('=' * 55)
print('  EXPORT COMPLETE')
print('=' * 55)
print(f'  Saved -> {OUTPUT_CSV}')
print(f'  Rows  : {len(wide)}')
print(f'  Cols  : {len(wide.columns)}')
print(f'  Columns: {list(wide.columns)}')

  EXPORT COMPLETE
  Saved -> adriatic_coast_stations_aggregated.csv
  Rows  : 122
  Cols  : 17
  Columns: ['station_id', 'sensor_name', 'lat', 'lon', 'codseqst', 'point', 'quota', 'gestore', 'provincia', 'TARIA2M_value', 'TARIA2M_dt', 'PRESS_value', 'PRESS_dt', 'VV_value', 'VV_dt', 'PREC_value', 'PREC_dt']


In [8]:
import folium

NETWORK_COLORS = {
    'MeteoHub_dpcn-veneto': 'blue',
    'MeteoHub_dpcn-marche': 'green',
    'MeteoHub_dpcn-puglia': 'red',
}
DEFAULT_COLOR = 'gray'

valid_coords = wide.dropna(subset=['lat', 'lon'])
map_center = [valid_coords['lat'].mean(), valid_coords['lon'].mean()]

m = folium.Map(location=map_center, zoom_start=7, tiles='CartoDB positron')

for _, row in valid_coords.iterrows():
    color = NETWORK_COLORS.get(row['gestore'], DEFAULT_COLOR)

    tooltip_lines = [f"<b>{row['sensor_name']}</b>", f"Rete: {row['gestore']}"]
    for code in SENSOR_CODES:
        col = f'{code}_value'
        if col in row and pd.notna(row[col]):
            unit = {'TARIA2M': '°C', 'PRESS': 'hPa', 'VV': 'm/s', 'PREC': 'mm'}.get(code, '')
            tooltip_lines.append(f"{code}: {row[col]:.1f} {unit}")
    tooltip_html = '<br>'.join(tooltip_lines)

    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        tooltip=tooltip_html,
    ).add_to(m)

# Legenda manuale
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
            background: white; padding: 10px; border-radius: 6px;
            box-shadow: 0 1px 4px rgba(0,0,0,0.3); font-size: 13px;">
  <b>Rete</b><br>
  <span style="color:blue;">●</span> Veneto<br>
  <span style="color:green;">●</span> Marche<br>
  <span style="color:red;">●</span> Puglia
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

print(f'Plotted {len(valid_coords)} / {len(wide)} stations (with valid coordinates)')
display(m)

Plotted 122 / 122 stations (with valid coordinates)
